# Fine-tuning of DeBERTa-v3-base
CSV in → 9 independent labels probability out

In [ ]:
# %pip install "transformers>=4.41" datasets accelerate sentencepiece protobuf scikit-learn

import json
import math
import os

import numpy as np
import torch
import torch.nn.functional as F
from datasets import Sequence, Value, load_dataset
from sklearn.metrics import f1_score
from torch import nn
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)


## Config

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
DATA_PATH = "data/og_aggr/"
TRAIN_CSV = DATA_PATH + "train.csv"
VAL_CSV = DATA_PATH + "val.csv"
TEXT_COL = "text"
LABEL_COLS = [
    "Joy",
    "Trust",
    "Fear",
    "Surprise",
    "Sadness",
    "Disgust",
    "Anger",
    "Anticipation",
    "Neutral",
]  # wide layout: one 0/1 column per label
LABEL_LIST_COL = None  # compact layout instead, e.g. "labels" holding "a|b"
MAX_LENGTH = 256  # ~95-99th percentile of token lengths, <= 512

LR = 1e-5
HEAD_LR = 1e-4
LAYER_DECAY = 0.9  # top-layer LR, new head LR, per-layer LR decay
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1
BATCH_SIZE = 64
GRAD_ACCUM = 1
EPOCHS = 4
HEAD_DROPOUT = 0.2
USE_POS_WEIGHT = False
MAX_POS_WEIGHT = 5.0  # up-weight rare positive labels
SEED = 42
OUTPUT_DIR = "runs/notebook"  # papermill overrides this with -p OUTPUT_DIR ...

set_seed(SEED)

## Dataset from CSV

In [ ]:
def _missing(x):
    return x is None or (isinstance(x, float) and math.isnan(x))

def build_multilabel_dataset(tokenizer, train_csv, val_csv=None, text_col="text",
                             label_cols=None, label_list_col=None, sep="|",
                             max_length=256, val_fraction=0.15, seed=42):
    if val_csv is None:
        raise ValueError('A validation set is required for training.')
    files = {"train": train_csv, "validation": val_csv}
    ds = load_dataset("csv", data_files=files)

    def encode(batch):
            enc = tokenizer(batch[text_col], truncation=True, max_length=max_length)
            enc["labels"] = [list(row) for row in zip(*(batch[c] for c in label_cols))]
            return enc

    ds = ds.map(encode, batched=True, remove_columns=ds["train"].column_names)
    return ds.cast_column("labels", Sequence(Value("float32")))  # float targets for BCE

def describe(ds, labels):
    for name, split in ds.items():
        y = np.asarray(split["labels"])
        print(f"{name}: {len(y)} rows, {int((y.sum(1) == 0).sum())} with no labels")
        for label, rate, n in zip(labels, y.mean(0), y.sum(0).astype(int)):
            print(f"  {label:<20} {rate:6.1%} ({n}){'  <-- no positives!' if n == 0 else ''}")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
CUSTOM_TOKENS = {
    "additional_special_tokens": [
        "[NAME]", # from GoEmotions
        "[RELIGION]",
        "URL", # from SemEval
    ]
}
tok.add_special_tokens(CUSTOM_TOKENS)

ds = build_multilabel_dataset(
    tok, TRAIN_CSV, VAL_CSV, TEXT_COL, LABEL_COLS, max_length=MAX_LENGTH, seed=SEED
)

describe(ds, LABEL_COLS)

lens = np.array([len(x) for x in ds["train"]["input_ids"]])
print(
    f"{(lens >= MAX_LENGTH).mean():.1%} of training texts hit MAX_LENGTH and were truncated"
)

# pos_weight from TRAIN only: negatives / positives per label, clipped.
y_train = np.asarray(ds["train"]["labels"])
pos = y_train.sum(0)
pos_weight = (
    np.clip((len(y_train) - pos) / np.maximum(pos, 1), 1.0, MAX_POS_WEIGHT)
    if USE_POS_WEIGHT
    else None
)

## Model

In [ ]:
class HeadedDeBERTa(nn.Module):
    def __init__(self, num_labels, pos_weight=None, vocab_size=None):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(MODEL_NAME, hidden_dropout_prob=0.15).float()
        if vocab_size and vocab_size > self.backbone.config.vocab_size:  # room for added tokens
            self.backbone.resize_token_embeddings(vocab_size)
        self.dropout = nn.Dropout(HEAD_DROPOUT)
        self.head = nn.Linear(self.backbone.config.hidden_size, num_labels)  # 768 -> K
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        pw = None if pos_weight is None else torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pw)

    def embed(self, input_ids, attention_mask):
        """Masked mean pooling -> raw 768-d vector per text."""
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-6)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        logits = self.head(self.dropout(self.embed(input_ids, attention_mask))).float()
        out = {"logits": logits}
        if labels is not None:
            out["loss"] = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=self.pos_weight)
        return out

In [ ]:
def build_optimizer(model):
    """AdamW with layer-wise LR decay; no weight decay on biases/LayerNorm."""
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    groups = []

    def add(named, lr):
        named = list(named)
        for wd, is_no_decay in ((WEIGHT_DECAY, False), (0.0, True)):
            ps = [p for n, p in named if any(nd in n for nd in no_decay) == is_no_decay]
            if ps:
                groups.append({"params": ps, "lr": lr, "weight_decay": wd})

    add(model.head.named_parameters(), HEAD_LR)
    enc, lr = model.backbone.encoder, LR
    for layer in reversed(enc.layer):                 # top layer first
        add(layer.named_parameters(), lr)
        lr *= LAYER_DECAY
    add(((n, p) for n, p in enc.named_parameters() if not n.startswith("layer.")), LR)
    add(model.backbone.embeddings.named_parameters(), lr)  # lowest LR

    assert sum(p.numel() for g in groups for p in g["params"]) == \
           sum(p.numel() for p in model.parameters()), "parameters missing from optimizer"
    return torch.optim.AdamW(groups, eps=1e-6)

## Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = (logits >= 0).astype(int)  # logit >= 0  <=>  sigmoid >= 0.5
    return {"f1_micro": f1_score(y, preds, average="micro", zero_division=0),
            "f1_macro": f1_score(y, preds, average="macro", zero_division=0)}

def tune_thresholds(probs, y, grid=np.arange(0.05, 0.96, 0.05)):
    """Best F1 threshold per label on validation (optimistic: confirm on a test set)."""
    return [float(grid[np.argmax([f1_score(y[:, j], probs[:, j] >= t, zero_division=0)
                                  for t in grid])])
            for j in range(y.shape[1])]

## Train

In [ ]:
model = HeadedDeBERTa(len(LABEL_COLS), pos_weight, vocab_size=len(tok))

steps_per_epoch = math.ceil(len(ds["train"]) / (BATCH_SIZE * GRAD_ACCUM))
eval_steps = max(1, steps_per_epoch // 2)                 # evaluate twice per epoch
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    warmup_steps=int(WARMUP_FRACTION * steps_per_epoch * EPOCHS),
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    eval_strategy="steps", eval_steps=eval_steps,
    save_strategy="steps", save_steps=eval_steps, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1_micro", greater_is_better=True,
    logging_steps=max(1, eval_steps // 5),
    bf16=bf16, fp16=torch.cuda.is_available() and not bf16,  # fp16 NaN? set fp16=False
    label_names=["labels"], seed=SEED, report_to="none",
    optim="adamw_torch", learning_rate=LR, weight_decay=WEIGHT_DECAY, adam_epsilon=1e-6,
    disable_tqdm=True,  # keeps the executed notebook and SLURM log readable
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    data_collator=DataCollatorWithPadding(tok, pad_to_multiple_of=8),
    compute_metrics=compute_metrics,
    # optimizers=(build_optimizer(model), None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.evaluate()



## Tune thresholds and save

In [ ]:
logits = trainer.predict(ds["validation"]).predictions
probs = 1 / (1 + np.exp(-logits))
thresholds = tune_thresholds(probs, np.asarray(ds["validation"]["labels"]))
print(dict(zip(LABEL_COLS, thresholds)))

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_state.pt"))
tok.save_pretrained(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "labels.json"), "w") as f:
    json.dump({"labels": LABEL_COLS, "thresholds": thresholds, "max_length": MAX_LENGTH}, f, indent=2)

## Inference

In [ ]:
@torch.no_grad()
def predict(texts):
    model.eval()
    device = next(model.parameters()).device
    enc = tok(texts, truncation=True, max_length=MAX_LENGTH,
              padding=True, return_tensors="pt").to(device)
    probs = torch.sigmoid(model(enc["input_ids"], enc["attention_mask"])["logits"]).cpu().numpy()
    return [[l for l, p, t in zip(LABEL_COLS, row, thresholds) if p >= t] for row in probs], probs

trainer.evaluate(ds["train"].select(range(min(2000, len(ds["train"])))), metric_key_prefix="train")